# UTILS

In [1]:
def compute_split(y, tile_size, H, n_splits):
    """
    Определяет split по вертикали (по центру тайла)
    """
    center_y = y + tile_size // 2
    split_height = H // n_splits

    split_id = center_y // split_height

    # защита от выхода за границы
    return min(split_id, n_splits - 1)

In [2]:
import numpy as np

def extract_tiles(data, mask, valid, tile_size=128, stride=32):
    """
    data: np.ndarray
        shape: (C, H, W) or (H, W, C)
    mask: (H, W)
    valid: (H, W)

    return: list of dictionaries with tiles
    """

    # Convert to (C, H, W)
    if data.ndim == 2:
        data = data[None, ...]
    elif data.shape[-1] in [1, 3]:
        data = np.transpose(data, (2, 0, 1))  # HWC → CHW

    C, H, W = data.shape

    tiles = []
    tile_id = 0

    for y in range(0, H - tile_size + 1, stride):
        for x in range(0, W - tile_size + 1, stride):

            tiles.append({
                "tile_id": tile_id,
                "y": y,
                "x": x,
                "tile_size": tile_size,

                "data": data[:, y:y+tile_size, x:x+tile_size],
                "mask": mask[y:y+tile_size, x:x+tile_size],
                "valid": valid[y:y+tile_size, x:x+tile_size],
            })

            tile_id += 1

    return tiles

In [3]:
import pandas as pd
import numpy as np

def build_manifest(
    tiles,
    H,
    n_splits=5,
    validation_split=4,
    full_map=False
):
    """
    tiles: список из extract_tiles()
    H: высота исходного изображения
    n_splits: общее количество сплитов
    validation_split: какой сплит использовать как val
    """

    rows = []

    for tile in tiles:
        mask = tile["mask"]
        valid = tile["valid"]

        total_pixels = mask.size
        valid_pixels = np.sum(valid)

        if valid_pixels == 0:
            continue

        positive_pixels = np.sum(mask * valid)

        valid_fraction = valid_pixels / total_pixels
        positive_fraction = positive_pixels / valid_pixels

        # --- вычисляем split ---
        split_id = compute_split(
            tile["y"],
            tile["tile_size"],
            H,
            n_splits
        ) if not full_map else 0

        subset = "full" if full_map else "val" if split_id == validation_split else "train"

        rows.append({
            "tile_id": tile["tile_id"],
            "y": tile["y"],
            "x": tile["x"],
            "tile_size": tile["tile_size"],

            "split_id": int(split_id),
            "subset": subset,

            "valid_fraction": float(valid_fraction),
            "positive_fraction": float(positive_fraction),
            "has_positive": int(positive_pixels > 0)
        })

    return pd.DataFrame(rows)

# Train splits

In [32]:
n_splits = 5
split_number = 5

data_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/processed/DEM_raw_shadowed.npz"
save_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/processed/"

In [ ]:
import numpy as np

_data = np.load(data_path)
data = _data["data"].astype(np.float32)
valid = _data["mask"].astype(np.uint8)
mask = _data["valid"].astype(np.uint8)

tiles = extract_tiles(data, mask, valid, tile_size=128, stride=32)

H = data.shape[-2] if data.ndim == 3 else data.shape[0]

In [33]:
manifest = build_manifest(
    tiles,
    H=H,
    n_splits=n_splits,
    validation_split=split_number
)

manifest.to_parquet(
    save_path+f"split{split_number}_manifest.parquet",
    index=False,
    engine="pyarrow"
)

In [23]:
df = pd.read_parquet(save_path+f"split{split_number}_manifest.parquet")

In [24]:
df["subset"].value_counts()

subset
train    126462
val       32314
Name: count, dtype: int64

In [25]:
df["split_id"].value_counts()

split_id
1    32314
2    32314
3    32292
4    31072
0    30784
Name: count, dtype: int64

In [26]:
df["has_positive"].value_counts(normalize=True)

has_positive
0    0.833035
1    0.166965
Name: proportion, dtype: float64

In [27]:
df["valid_fraction"].describe()

count    158776.000000
mean          0.989549
std           0.080366
min           0.000122
25%           1.000000
50%           1.000000
75%           1.000000
max           1.000000
Name: valid_fraction, dtype: float64

# Full split

In [4]:
n_splits = 0
split_number = 0

data_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/processed/DEM_raw_shadowed.npz"
save_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/processed/"

In [5]:
import numpy as np

_data = np.load(data_path)
data = _data["data"].astype(np.float32)
valid = _data["mask"].astype(np.uint8)
mask = _data["valid"].astype(np.uint8)

tiles = extract_tiles(data, mask, valid, tile_size=128, stride=32)

H = data.shape[-2] if data.ndim == 3 else data.shape[0]

In [7]:
manifest = build_manifest(
    tiles,
    H=H,
    n_splits=n_splits,
    validation_split=split_number,
    full_map=True
)

manifest.to_parquet(
    save_path+"full_map_manifest.parquet",
    index=False,
    engine="pyarrow"
)

In [8]:
df = pd.read_parquet(save_path+"full_map_manifest.parquet")

In [9]:
df.describe

<bound method NDFrame.describe of         tile_id      y     x  tile_size  split_id subset  valid_fraction  \
0             0      0     0        128         0   full        0.108948   
1             1      0    32        128         0   full        0.255432   
2             2      0    64        128         0   full        0.402832   
3             3      0    96        128         0   full        0.551270   
4             4      0   128        128         0   full        0.590759   
...         ...    ...   ...        ...       ...    ...             ...   
158771   160355  16960  9440        128         0   full        0.976257   
158772   160356  16960  9472        128         0   full        0.761108   
158773   160357  16960  9504        128         0   full        0.511108   
158774   160358  16960  9536        128         0   full        0.261108   
158775   160359  16960  9568        128         0   full        0.034851   

        positive_fraction  has_positive  
0          